# Chapter 07 — Pandas II: Cleaning, Transformation, and Aggregation

Real datasets are rarely analysis-ready. Learn missing-value handling, duplicate detection, strings, dates, grouping, joins, and reshaping.

## Prerequisites
Chapter 6 DataFrame selection, dtypes, and inspection.

## Learning goals
- Diagnose missing values and duplicates.
- Use string, datetime, and categorical accessors.
- Transform with vectorized operations, map, and apply.
- Group, aggregate, merge, concatenate, pivot, and melt.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Chapter map

1. Missing values
2. Duplicates and strings
3. Dates and times
4. GroupBy and aggregation
5. Merge and concat
6. Reshaping

### Missing values
Pandas uses `np.nan`, `pd.NA`, and `NaT`. Use `isna`/`notna` for detection. `dropna(axis=0, how="any", subset=None, thresh=None)` removes rows/columns based on missingness. `fillna(value, limit=...)` accepts scalar, mapping, Series, or DataFrame values. Forward/backward filling is meaningful only when row order has temporal meaning. Decide whether missingness itself carries information.

#### Demonstration 1 — Diagnose a messy dataset

Run the code and inspect both the output and the assumptions behind it.

In [2]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())
import pandas as pd
dirty=pd.read_csv(DATA/"messy_sales.csv"); print(dirty.isna().sum()); print("duplicate full rows:",dirty.duplicated().sum())

Data directory: /mnt/data/python_data_analysis_book/data
order_id       0
order_date     0
region         7
product        0
units          6
unit_price     0
discount       0
revenue        0
cost           0
profit         0
customer_id    0
dtype: int64
duplicate full rows: 2


### Duplicates and strings
`duplicated(subset=None, keep="first")` marks duplicate rows; `keep="last"` marks earlier copies; `keep=False` marks all members. `drop_duplicates(subset=None, keep="first", ignore_index=False)` removes duplicates. Define identity using business keys. The `.str` accessor includes `strip`, `lower`, `upper`, `replace`, `contains`, `startswith`, `len`, and `extract`; use `na=` where supported.

#### Demonstration 2 — Clean text and deduplicate

Run the code and inspect both the output and the assumptions behind it.

In [3]:
dirty["product_clean"]=dirty["product"].astype("string").str.strip().str.title()
deduped=dirty.drop_duplicates(subset=["order_id"],keep="first")
print(dirty.product_clean.value_counts(dropna=False)); print("rows:",len(dirty),"to",len(deduped))

product_clean
Webcam      16
Keyboard    16
Mouse       13
Laptop       9
Monitor      8
Name: count, dtype: Int64
rows: 62 to 60


### Dates and times
`pd.to_datetime(arg, errors="raise", utc=False, format=None)` parses timestamps. `errors="coerce"` turns invalid values into `NaT`. `.dt` exposes year, month, day, quarter, and day name. `dt.to_period("M")` produces monthly periods; `dt.floor("D")` rounds down. Set a consistent timezone policy; UTC normalization may require local conversion for reporting.

#### Demonstration 3 — Parse dates and group by month

Run the code and inspect both the output and the assumptions behind it.

In [4]:
sales=pd.read_csv(DATA/"sales.csv",parse_dates=["order_date"])
sales["month"]=sales.order_date.dt.to_period("M")
monthly=sales.groupby("month",as_index=False).agg(revenue=("revenue","sum"),orders=("order_id","count"))
display(monthly.head())

     month     revenue  orders
0  2025-01  2442695.25      18
1  2025-02  3403578.19      22
2  2025-03  2936413.06      20
3  2025-04  2785406.87      22
4  2025-05  7168038.71      27


### GroupBy and aggregation
`groupby(by, as_index=True, sort=True, dropna=True)` splits rows into groups. `.agg()` computes summaries; `.transform()` returns an aligned result; `.filter()` retains groups. Named aggregation such as `revenue_total=("revenue","sum")` gives stable names. Prefer built-in operations over Python `apply` where possible.

#### Demonstration 4 — Aggregate multiple dimensions

Run the code and inspect both the output and the assumptions behind it.

In [5]:
regional=(sales.groupby(["region","product"],as_index=False).agg(orders=("order_id","count"),units=("units","sum"),revenue=("revenue","sum"),profit=("profit","sum")))
display(regional.head())

  region   product  orders  units     revenue      profit
0   East  Keyboard      12    122   252174.34    72899.05
1   East    Laptop      10    110  6523684.75  1166941.66
2   East   Monitor      10    100  1356327.71   429207.02
3   East     Mouse      15    132   108817.76    23630.60
4   East    Webcam      16    150   466065.20   118974.22


### Merge and concat
`pd.merge(left,right,how="inner",on=None,left_on=None,right_on=None,suffixes=("_x","_y"),validate=None,indicator=False)` joins relational keys. `inner`, `left`, `right`, and `outer` have different retention rules. `validate` can assert one-to-one, one-to-many, or many-to-one. `pd.concat(objs,axis=0,join="outer",ignore_index=False,keys=None)` stacks along rows or columns; use merge for keyed joins and concat for stacking.

#### Demonstration 5 — Join customer context and check cardinality

Run the code and inspect both the output and the assumptions behind it.

In [6]:
customers=pd.read_csv(DATA/"customers.csv"); orders=pd.read_csv(DATA/"orders.csv")
joined=orders.merge(customers,on="customer_id",how="left",validate="many_to_one",indicator=True)
print("merge status:",joined._merge.value_counts().to_dict(),"rows:",len(joined))

merge status: {'both': 240, 'left_only': 0, 'right_only': 0} rows: 240


### Reshaping
`pivot(index,columns,values)` requires unique key combinations. `pivot_table` aggregates duplicates and supports `aggfunc`, `fill_value`, and `margins`. `melt(id_vars,value_vars,var_name,value_name)` converts wide values to long form. Tidy/long data is convenient for plotting and statistical modeling.

#### Demonstration 6 — Pivot and melt

Run the code and inspect both the output and the assumptions behind it.

In [7]:
wide=sales.pivot_table(index="region",columns="product",values="revenue",aggfunc="sum",fill_value=0)
display(wide); long=wide.reset_index().melt(id_vars="region",var_name="product",value_name="revenue"); display(long.head())

product   Keyboard       Laptop     Monitor      Mouse     Webcam
region                                                           
East     252174.34   6523684.75  1356327.71  108817.76  466065.20
North    161154.07   6027535.00  1631297.62   71486.13  270342.08
South    357595.68   8186765.07  1182551.26   52404.13  400667.42
West     372185.97  10383230.12  1312339.75  102002.24  200883.36
  region   product     revenue
0   East  Keyboard   252174.34
1  North  Keyboard   161154.07
2  South  Keyboard   357595.68
3   West  Keyboard   372185.97
4   East    Laptop  6523684.75


## Self-check

Without looking back, explain the key terms, identify the important parameters, describe at least one alternative or variation, and state one limitation of the methods covered. Then answer the exercises below.

### Review questions

- Explain one important concept from: Missing values.
- Explain one important concept from: Duplicates and strings.
- Explain one important concept from: Dates and times.
- Explain one important concept from: GroupBy and aggregation.
- Explain one important concept from: Merge and concat.